In [1]:
import time
import numpy as np
import pandas as pd
import openml
from pymfe.mfe import MFE
import warnings

suite = openml.study.get_suite("OpenML-CC18")
tasks = openml.tasks.list_tasks(task_id=suite.tasks, output_format="dataframe")
print(len(tasks))
print(tasks.columns.tolist())

72
['tid', 'ttid', 'did', 'name', 'task_type', 'status', 'estimation_procedure', 'source_data', 'target_feature', 'MajorityClassSize', 'MaxNominalAttDistinctValues', 'MinorityClassSize', 'NumberOfClasses', 'NumberOfFeatures', 'NumberOfInstances', 'NumberOfInstancesWithMissingValues', 'NumberOfMissingValues', 'NumberOfNumericFeatures', 'NumberOfSymbolicFeatures']


In [2]:
print(tasks.estimation_procedure.value_counts())

selected_tasks = tasks[
    tasks.NumberOfInstances.between(500, 50000) &
    (tasks.NumberOfFeatures <= 500)
]

print(len(selected_tasks))

tam = selected_tasks.assign(tam=selected_tasks.NumberOfInstances * selected_tasks.NumberOfFeatures).sort_values("tam")
amostra = tam.iloc[[0, len(tam) // 2, -1]]

def pegar_dados(did, alvo, tentativas=3):
    for k in range(tentativas):
        try:
            ds = openml.datasets.get_dataset(int(did))
            return ds.get_data(target=alvo, dataset_format="dataframe")
        except Exception as e:
            print("   falha ao baixar, tentando de novo:", repr(e)[:120])
            time.sleep(10)
    raise RuntimeError("não consegui baixar o dataset")

def testar_pymfe(linha):
    X, y, cat_ind, _ = pegar_dados(linha.did, linha.target_feature)
    X = X.copy()
    cat_cols = [i for i, c in enumerate(cat_ind) if c]
    for i in cat_cols:
        col = X.columns[i]
        cod = X[col].astype("category").cat.codes.astype(float)
        X[col] = cod.where(cod >= 0)
    y_arr = pd.Series(y).astype("category").cat.codes.to_numpy()
    X_arr = X.to_numpy(dtype=float)

    t0 = time.time()
    mfe = MFE(groups=["general", "statistical", "info-theory"])
    mfe.fit(X_arr, y_arr, cat_cols=cat_cols)
    nomes, valores = mfe.extract()
    return len(nomes), time.time() - t0

for linha in amostra.itertuples():
    print(linha.name, "|", linha.NumberOfInstances, "instâncias |", linha.NumberOfFeatures, "features")
    try:
        n, seg = testar_pymfe(linha)
        print("   ", n, "metacaracterísticas em", f"{seg:.1f}s")
    except Exception as e:
        print("    ERRO:", repr(e)[:300])

estimation_procedure
10-fold Crossvalidation    72
Name: count, dtype: int64
60
balance-scale | 625 instâncias | 5 features


c:\Users\caioc\recomendacao-classificadores\.venv\Lib\site-packages\pymfe\_internal.py:1567: UserWarning: It is not possible make equal discretization
  warnings.warn("It is not possible make equal discretization")
c:\Users\caioc\recomendacao-classificadores\.venv\Lib\site-packages\pymfe\_internal.py:1567: UserWarning: It is not possible make equal discretization
  warnings.warn("It is not possible make equal discretization")


    73 metacaracterísticas em 0.1s
analcatdata_authorship | 841 instâncias | 71 features


c:\Users\caioc\recomendacao-classificadores\.venv\Lib\site-packages\pymfe\statistical.py:1266: RuntimeWarning: invalid value encountered in log
  np.log(np.linalg.det(S_i)) for S_i in sample_cov_matrices
c:\Users\caioc\recomendacao-classificadores\.venv\Lib\site-packages\pymfe\statistical.py:1271: RuntimeWarning: invalid value encountered in log
  * np.log(np.linalg.det(pooled_cov_mat))


    73 metacaracterísticas em 1.3s
nomao | 34465 instâncias | 119 features


c:\Users\caioc\recomendacao-classificadores\.venv\Lib\site-packages\pymfe\_internal.py:1567: UserWarning: It is not possible make equal discretization
  warnings.warn("It is not possible make equal discretization")
c:\Users\caioc\recomendacao-classificadores\.venv\Lib\site-packages\pymfe\_internal.py:732: RuntimeWarning: Can't summarize feature 'can_cor' with summary 'sd'. Will set it as 'np.nan'.
  warnings.warn(
c:\Users\caioc\recomendacao-classificadores\.venv\Lib\site-packages\scipy\stats\_morestats.py:2199: RuntimeWarning: invalid value encountered in scalar divide
  return num / den
c:\Users\caioc\recomendacao-classificadores\.venv\Lib\site-packages\pymfe\statistical.py:1266: RuntimeWarning: divide by zero encountered in log
  np.log(np.linalg.det(S_i)) for S_i in sample_cov_matrices
c:\Users\caioc\recomendacao-classificadores\.venv\Lib\site-packages\pymfe\statistical.py:1271: RuntimeWarning: divide by zero encountered in log
  * np.log(np.linalg.det(pooled_cov_mat))
c:\Users\cai

    73 metacaracterísticas em 8.1s


In [3]:
rng = np.random.default_rng(0)

def mf_com_codigos(linha, permutar):
    X, y, cat_ind, _ = pegar_dados(linha.did, linha.target_feature)
    X = X.copy()
    cat_cols = [i for i, c in enumerate(cat_ind) if c]
    for i in cat_cols:
        col = X.columns[i]
        cod = X[col].astype("category").cat.codes.to_numpy()
        if permutar:
            perm = rng.permutation(cod.max() + 1)
            cod = np.where(cod >= 0, perm[np.clip(cod, 0, None)], -1)
        X[col] = np.where(cod >= 0, cod, np.nan).astype(float)
    y_arr = pd.Series(y).astype("category").cat.codes.to_numpy()

    mfe = MFE(groups=["general", "statistical", "info-theory"])
    mfe.fit(X.to_numpy(dtype=float), y_arr, cat_cols=cat_cols)
    nomes, valores = mfe.extract()
    return pd.Series(valores, index=nomes, dtype=float)

linha = next(selected_tasks[selected_tasks.name == "car"].itertuples())
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    a  = mf_com_codigos(linha, permutar=False)
    a2 = mf_com_codigos(linha, permutar=False)
    b  = mf_com_codigos(linha, permutar=True)

def mudou(x, z):
    return (x.isna() != z.isna()) | ((x - z).abs() > 1e-9)

print("controle (a vs a2):", mudou(a, a2).sum(), "de", len(a), "mudaram")
print("embaralhado (a vs b):", mudou(a, b).sum(), "de", len(a), "mudaram")
print(a[mudou(a, b)].index.tolist())

controle (a vs a2): 0 de 73 mudaram
embaralhado (a vs b): 1 de 73 mudaram
['gravity']


In [4]:
def mf_freq(linha):
    X, y, cat_ind, _ = pegar_dados(linha.did, linha.target_feature)
    X = X.copy()
    cat_cols = [i for i, c in enumerate(cat_ind) if c]
    for i in cat_cols:
        col = X.columns[i]
        freq = X[col].value_counts(normalize=True).to_dict()
        X[col] = X[col].astype(object).map(freq).astype(float)
    y_arr = pd.Series(y).astype("category").cat.codes.to_numpy()
    mfe = MFE(groups=["general", "statistical", "info-theory"])
    mfe.fit(X.to_numpy(dtype=float), y_arr, cat_cols=cat_cols)
    nomes, valores = mfe.extract()
    return pd.Series(valores, index=nomes, dtype=float)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    f = mf_freq(linha)

dif = mudou(a, f)
print(dif.sum(), "de", len(a), "diferem da codificação ordinal")
print(a[dif].index.tolist())

56 de 73 diferem da codificação ordinal
['attr_conc.mean', 'attr_conc.sd', 'attr_ent.mean', 'attr_ent.sd', 'can_cor.mean', 'can_cor.sd', 'class_conc.mean', 'class_conc.sd', 'cor.mean', 'cor.sd', 'cov.mean', 'cov.sd', 'eigenvalues.mean', 'eigenvalues.sd', 'eq_num_attr', 'g_mean.mean', 'g_mean.sd', 'gravity', 'h_mean.mean', 'h_mean.sd', 'iq_range.mean', 'iq_range.sd', 'joint_ent.mean', 'joint_ent.sd', 'kurtosis.mean', 'kurtosis.sd', 'lh_trace', 'mad.sd', 'max.sd', 'mean.mean', 'mean.sd', 'median.mean', 'median.sd', 'min.mean', 'min.sd', 'mut_inf.mean', 'mut_inf.sd', 'nr_cor_attr', 'nr_disc', 'nr_outliers', 'ns_ratio', 'p_trace', 'range.mean', 'range.sd', 'roy_root', 'sd.mean', 'sd.sd', 'skewness.mean', 'skewness.sd', 'sparsity.mean', 'sparsity.sd', 't_mean.mean', 't_mean.sd', 'var.mean', 'var.sd', 'w_lambda']


In [5]:
print(selected_tasks.loc[selected_tasks.name == "car", "NumberOfInstances"].iloc[0], "=", 4*4*4*3*3*3)
X, y, cat_ind, _ = pegar_dados(linha.did, linha.target_feature)
print(X.iloc[:, 0].value_counts())

1728 = 1728
buying
vhigh    432
high     432
med      432
low      432
Name: count, dtype: int64


In [6]:
cand = selected_tasks[selected_tasks.name == "credit-g"]
print(len(cand), "base(s) encontrada(s)")
linha2 = next(cand.itertuples())

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    a_  = mf_com_codigos(linha2, permutar=False)
    a2_ = mf_com_codigos(linha2, permutar=False)
    b_  = mf_com_codigos(linha2, permutar=True)
    f_  = mf_freq(linha2)

print("controle      :", mudou(a_, a2_).sum(), "de", len(a_))
print("embaralhado   :", mudou(a_, b_).sum(), "de", len(a_))
print("frequência    :", mudou(a_, f_).sum(), "de", len(a_))
print(a_[mudou(a_, b_)].index.tolist())

1 base(s) encontrada(s)
controle      : 2 de 73
embaralhado   : 27 de 73
frequência    : 56 de 73
['attr_conc.mean', 'attr_conc.sd', 'cor.mean', 'cor.sd', 'cov.mean', 'cov.sd', 'eigenvalues.mean', 'eigenvalues.sd', 'gravity', 'iq_range.mean', 'iq_range.sd', 'kurtosis.mean', 'kurtosis.sd', 'mean.mean', 'mean.sd', 'median.mean', 'median.sd', 'nr_cor_attr', 'nr_outliers', 'sd.mean', 'sd.sd', 'skewness.mean', 'skewness.sd', 't_mean.mean', 't_mean.sd', 'var.mean', 'var.sd']


In [7]:
print(MFE.fit.__doc__)

Fits dataset into an MFE model.

Parameters
----------
X : :obj:`List`
    Predictive attributes of the dataset.

y : :obj:`List`, optional
    Target attributes of the dataset, assuming that it is a supervised
    task.

transform_num : :obj:`bool`, optional
    If True, numeric attributes are discretized using equal-frequency
    histogram technique to use alongside categorical data when
    extracting categoric-only metafeatures. Note that numeric-only
    features still uses the original numeric values, not the
    discretized ones. If False, then numeric attributes are ignored for
    categorical-only meta-features.

transform_cat : :obj:`str`, optional
    Transform categorical data to use alongside numerical data while
    extracting numeric-only metafeatures. Note that categoric-only
    features still uses the original categoric values, and not the
    binarized ones.

    If `one-hot`, categorical attributes are binarized using one-hot
    encoding with `k-1` features for a c

In [8]:
def mf_nativo(linha, transform_cat="gray", renomear=False):
    X, y, cat_ind, _ = pegar_dados(linha.did, linha.target_feature)
    X = X.copy()
    cat_cols = [i for i, c in enumerate(cat_ind) if c]
    for i in cat_cols:
        col = X.columns[i]
        s = X[col].astype(object)
        if renomear:
            cats = list(s.dropna().unique())
            novo = dict(zip(cats, rng.permutation(len(cats))))
            s = s.map(lambda v: f"c{novo[v]}" if v in novo else v)
        X[col] = s
    y_arr = pd.Series(y).astype("category").cat.codes.to_numpy()
    mfe = MFE(groups=["general", "statistical", "info-theory"])
    mfe.fit(X.to_numpy(dtype=object), y_arr, cat_cols=cat_cols, transform_cat=transform_cat)
    nomes, valores = mfe.extract()
    return pd.Series(valores, index=nomes, dtype=float)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    g1 = mf_nativo(linha2, "gray")
    g2 = mf_nativo(linha2, "gray")
    g3 = mf_nativo(linha2, "gray", renomear=True)
    oh = mf_nativo(linha2, "one-hot")

print("controle (gray vs gray)     :", mudou(g1, g2).sum(), "de", len(g1))
print("renomeado (gray vs gray')   :", mudou(g1, g3).sum(), "de", len(g1))
print("gray vs one-hot             :", mudou(g1, oh).sum(), "de", len(g1))
print("gray vs ordinal (a_)        :", mudou(g1, a_).sum(), "de", len(g1))
print(g1[mudou(g1, g3)].index.tolist())

controle (gray vs gray)     : 2 de 73
renomeado (gray vs gray')   : 27 de 73
gray vs one-hot             : 39 de 73
gray vs ordinal (a_)        : 27 de 73
['attr_conc.mean', 'attr_conc.sd', 'cor.mean', 'cor.sd', 'cov.mean', 'cov.sd', 'eigenvalues.mean', 'eigenvalues.sd', 'gravity', 'iq_range.mean', 'iq_range.sd', 'kurtosis.mean', 'kurtosis.sd', 'mean.mean', 'mean.sd', 'median.mean', 'median.sd', 'nr_cor_attr', 'nr_outliers', 'sd.mean', 'sd.sd', 'skewness.mean', 'skewness.sd', 't_mean.mean', 't_mean.sd', 'var.mean', 'var.sd']


In [9]:
com_cat = selected_tasks.NumberOfSymbolicFeatures > 1
print(com_cat.sum(), "de", len(selected_tasks), "bases têm atributos categóricos")
print(selected_tasks.loc[com_cat, ["name", "NumberOfSymbolicFeatures", "NumberOfNumericFeatures"]]
      .sort_values("NumberOfSymbolicFeatures", ascending=False).head(10))

rng = np.random.default_rng(0)

def mudou(x, z):
    return (x.isna() != z.isna()) | ((x - z).abs() > 1e-9)

def resumo(x, z, rotulo):
    m = mudou(x, z)
    ok = m & x.notna() & z.notna()
    rel = ((x - z).abs() / x.abs().clip(lower=1e-12))[ok]
    print(f"   {rotulo:<10}: {int(m.sum()):>2} de {len(x)} mudaram | "
          f"mudança relativa mediana {rel.median():.1%} | acima de 10%: {int((rel > 0.10).sum())}")

for tc in ["gray", "one-hot-full", None]:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        u1  = mf_nativo(linha2, tc)
        u1b = mf_nativo(linha2, tc)                  
        u2  = mf_nativo(linha2, tc, renomear=True)
    print(f"\ntransform_cat = {tc}")
    resumo(u1, u1b, "controle")
    resumo(u1, u2, "renomeado")
    print(f"   NaN: {int(u1.isna().sum())} | infinitos: {int(np.isinf(u1).sum())}")
    print("   instáveis no controle:", u1[mudou(u1, u1b)].index.tolist())

22 de 60 bases têm atributos categóricos
                    name  NumberOfSymbolicFeatures  NumberOfNumericFeatures
167140               dna                       181                        0
45                splice                        61                        0
3               kr-vs-kp                        37                        0
14952   PhishingWebsites                        31                        0
9977               nomao                        30                       89
3021                sick                        23                        7
14954     cylinder-bands                        22                       18
31              credit-g                        14                        7
125920     dresses-sales                        12                        1
29       credit-approval                        10                        6

transform_cat = gray
   controle  :  2 de 73 mudaram | mudança relativa mediana 94.2% | acima de 10%: 2
   renomeado : 24 

In [10]:
for nome in ["kr-vs-kp", "splice", "PhishingWebsites", "credit-approval"]:
    lin = next(selected_tasks[selected_tasks.name == nome].itertuples())
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            t0 = time.time()
            u1 = mf_nativo(lin, "one-hot-full")
            seg = time.time() - t0
            u1b = mf_nativo(lin, "one-hot-full")
            u2 = mf_nativo(lin, "one-hot-full", renomear=True)
    except Exception as e:
        print(f"\n{nome}: ERRO {repr(e)[:200]}")
        continue
    print(f"\n{nome} ({seg:.1f}s)")
    resumo(u1, u1b, "controle")
    resumo(u1, u2, "renomeado")
    print(f"   NaN: {int(u1.isna().sum())} | infinitos: {int(np.isinf(u1).sum())}")
    print("   NaN em:", u1[u1.isna()].index.tolist())


kr-vs-kp (1.4s)
   controle  :  2 de 73 mudaram | mudança relativa mediana 93.4% | acima de 10%: 2
   renomeado :  2 de 73 mudaram | mudança relativa mediana 166.9% | acima de 10%: 2
   NaN: 3 | infinitos: 0
   NaN em: ['can_cor.sd', 'cat_to_num', 'sd_ratio']

splice (2.4s)
   controle  :  2 de 73 mudaram | mudança relativa mediana 68.4% | acima de 10%: 2
   renomeado :  2 de 73 mudaram | mudança relativa mediana 53.0% | acima de 10%: 2
   NaN: 2 | infinitos: 0
   NaN em: ['cat_to_num', 'sd_ratio']

PhishingWebsites (1.8s)
   controle  :  2 de 73 mudaram | mudança relativa mediana 7.8% | acima de 10%: 0
   renomeado :  2 de 73 mudaram | mudança relativa mediana 25.6% | acima de 10%: 2
   NaN: 3 | infinitos: 0
   NaN em: ['can_cor.sd', 'cat_to_num', 'sd_ratio']

credit-approval (0.8s)
   controle  :  2 de 73 mudaram | mudança relativa mediana 12.3% | acima de 10%: 1
   renomeado :  2 de 73 mudaram | mudança relativa mediana 9.3% | acima de 10%: 1
   NaN: 52 | infinitos: 0
   NaN em: ['

In [11]:
print((selected_tasks.NumberOfMissingValues > 0).sum(), "de", len(selected_tasks),
      "bases têm valores faltantes")

def mf_final(linha, transform_cat="one-hot-full"):
    X, y, cat_ind, _ = pegar_dados(linha.did, linha.target_feature)
    X = X.copy()
    cat_cols = [i for i, c in enumerate(cat_ind) if c]
    for i, col in enumerate(X.columns):
        if i in cat_cols:
            s = X[col].astype(object)
            moda = s.mode(dropna=True)
            X[col] = s.fillna(moda.iloc[0] if len(moda) else "faltante")
        else:
            s = pd.to_numeric(X[col], errors="coerce")
            X[col] = s.fillna(s.median() if s.notna().any() else 0.0)
    y_arr = pd.Series(y).astype("category").cat.codes.to_numpy()
    mfe = MFE(groups=["general", "statistical", "info-theory"])
    mfe.fit(X.to_numpy(dtype=object), y_arr, cat_cols=cat_cols, transform_cat=transform_cat)
    nomes, valores = mfe.extract()
    return pd.Series(valores, index=nomes, dtype=float)

for nome in ["credit-approval", "dna"]:
    lin = next(selected_tasks[selected_tasks.name == nome].itertuples())
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            t0 = time.time()
            u = mf_final(lin)
            seg = time.time() - t0
        print(f"\n{nome}: {seg:.1f}s | NaN: {int(u.isna().sum())} | infinitos: {int(np.isinf(u).sum())}")
        print("   NaN em:", u[u.isna()].index.tolist())
    except Exception as e:
        print(f"\n{nome}: ERRO {repr(e)[:200]}")

9 de 60 bases têm valores faltantes

credit-approval: 0.9s | NaN: 2 | infinitos: 0
   NaN em: ['can_cor.sd', 'sd_ratio']

dna: 4.2s | NaN: 2 | infinitos: 0
   NaN em: ['cat_to_num', 'sd_ratio']
